# Two-dimensional normalising flow

Make invertibility, density correction, and the Jacobian determinant visible with two explicit transformations.

**Classroom pattern:** predict first, run one cell, inspect the intermediate state, then change one parameter. All random seeds are fixed for the first run.

## Before you run

- Use the environment from `README.md`; no lecturer-specific Conda environment is required.
- Set `LECTURE2_QUICK=1` before starting Jupyter for the CPU-safe verification path.
- CUDA is selected first, then Apple MPS, then CPU. A separate MLX installation is not required.
- Read the prediction prompt before each experiment. The fixed-seed result is a mechanism check, not a quality benchmark.
- Every generated file stays inside `fallbacks/` or `checkpoints/`. No cell uploads or publishes anything.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "demo_models.py").exists():
    candidate = ROOT / "lecture 2" / "demos"
    if candidate.exists():
        ROOT = candidate
sys.path.insert(0, str(ROOT))
(ROOT / "fallbacks").mkdir(exist_ok=True)
print(f"Demo root: {ROOT}")

import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(12)

## 1. Base samples and two invertible maps

The first map doubles the horizontal axis. The second adds a shear with determinant one.

In [ ]:
z = rng.normal(size=(1200, 2))

def scale_forward(points, scale=2.0):
    out = points.copy(); out[:, 0] *= scale
    return out

def scale_inverse(points, scale=2.0):
    out = points.copy(); out[:, 0] /= scale
    return out

def shear_forward(points, strength=0.9):
    out = points.copy(); out[:, 1] += strength * np.tanh(points[:, 0] / 2)
    return out

def shear_inverse(points, strength=0.9):
    out = points.copy(); out[:, 1] -= strength * np.tanh(points[:, 0] / 2)
    return out

x1 = scale_forward(z)
x2 = shear_forward(x1)

## 2. Geometry and density

A local horizontal expansion by 2 divides density by 2. The shear changes shape but has determinant 1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True, sharey=True)
for ax, points, title in zip(axes, [z, x1, x2], ["z ~ N(0,I)", "scale x-axis by 2", "invertible shear"]):
    ax.scatter(points[:, 0], points[:, 1], s=5, alpha=0.25)
    ax.axhline(0, color="0.8", lw=0.8); ax.axvline(0, color="0.8", lw=0.8)
    ax.set_title(title); ax.set_xlabel("coordinate 1")
axes[0].set_ylabel("coordinate 2")
fig.suptitle("Forward maps bend samples; the Jacobian tracks local volume")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "03_flow_warp.png", dpi=180, bbox_inches="tight")
plt.show()

log_pz_at_zero = -np.log(2 * np.pi)
log_det_scale = np.log(2.0)
print(f"log p_z(0) = {log_pz_at_zero:.3f}")
print(f"log |det J_scale| = {log_det_scale:.3f}")
print(f"log p_x(f(0)) = {log_pz_at_zero - log_det_scale:.3f}")
print("density ratio after the expansion:", np.exp(-log_det_scale))

## 3. Inverse check

Exact reversible sampling requires that the numerical inverse returns every point.

In [ ]:
recovered = scale_inverse(shear_inverse(x2))
error = np.max(np.abs(recovered - z))
print(f"maximum reconstruction error: {error:.3e}")
assert error < 1e-10

## 4. Train a RealNVP rather than hand-designing the warp

A RealNVP stacks affine coupling layers. Each layer leaves part of the vector unchanged and uses it to scale and shift the remaining coordinates. This triangular Jacobian makes the determinant cheap.

In [ ]:
import os
import torch
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display
from demo_models import RealNVP

torch.manual_seed(12)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
QUICK_MODE = os.getenv("LECTURE2_QUICK", "0") == "1"
artifact_suffix = "_smoke" if QUICK_MODE else ""
print("device:", device, "quick mode:", QUICK_MODE)

def make_two_moons(count=2000, noise=0.08, seed=12):
    generator = np.random.default_rng(seed)
    first = count // 2
    angle_a = generator.uniform(0, np.pi, first)
    angle_b = generator.uniform(0, np.pi, count - first)
    moon_a = np.c_[np.cos(angle_a), np.sin(angle_a)]
    moon_b = np.c_[1 - np.cos(angle_b), 0.45 - np.sin(angle_b)]
    points = np.vstack([moon_a, moon_b]) + generator.normal(0, noise, (count, 2))
    points = (points - points.mean(0)) / points.std(0)
    return torch.tensor(points, dtype=torch.float32)

target = make_two_moons(2400 if not QUICK_MODE else 600)
flow = RealNVP(dim=2, layers=6 if not QUICK_MODE else 4, hidden=64 if not QUICK_MODE else 32).to(device)
optimizer = torch.optim.Adam(flow.parameters(), lr=2e-3)
steps = 1500 if not QUICK_MODE else 180
fixed_z = torch.randn(800, 2, device=device)
frames, losses, frame_steps = [], [], []

for step in range(steps + 1):
    indices = torch.randint(0, len(target), (256,))
    batch = target[indices].to(device)
    loss = -flow.log_prob(batch).mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
    if step % max(steps // 12, 1) == 0 or step == steps:
        with torch.no_grad():
            generated, _ = flow(fixed_z)
        frames.append(generated.cpu().numpy())
        frame_steps.append(step)

print(f"initial NLL {losses[0]:.3f}; final NLL {losses[-1]:.3f}")

## 5. Source, target, and learned samples

The base density stays Gaussian. Training changes the invertible map so its pushforward approaches the two-moon target.

In [ ]:
base = fixed_z.detach().cpu().numpy()
target_np = target.numpy()
final_np = frames[-1]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True, sharey=True)
for ax, points, title, color in zip(
    axes,
    [base, target_np, final_np],
    ["source z ~ N(0,I)", "target data", "trained flow samples"],
    ["tab:purple", "0.45", "tab:blue"],
):
    ax.scatter(points[:, 0], points[:, 1], s=7, alpha=0.35, c=color)
    ax.set_title(title); ax.set_xlabel("dimension 1")
axes[0].set_ylabel("dimension 2")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / f"03_realnvp_trained{artifact_suffix}.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(target_np[:, 0], target_np[:, 1], s=5, alpha=0.14, c="black", label="target")
moving = ax.scatter([], [], s=8, alpha=0.45, c="tab:blue", label="flow samples")
ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2); ax.legend(loc="upper right")
def update(frame_index):
    moving.set_offsets(frames[frame_index])
    ax.set_title(f"RealNVP training step {frame_steps[frame_index]}")
    return (moving,)
animation = FuncAnimation(fig, update, frames=len(frames), interval=650, blit=False)
gif_path = ROOT / "fallbacks" / f"03_realnvp_training{artifact_suffix}.gif"
animation.save(gif_path, writer=PillowWriter(fps=2))
plt.close(fig)
display(HTML(animation.to_jshtml()))
print("saved", gif_path)

## 6. Forward and inverse after training

Sampling uses z → x. Likelihood evaluation uses x → z and adds the inverse log-determinant.

In [ ]:
with torch.no_grad():
    test_z = torch.randn(128, 2, device=device)
    test_x, forward_logdet = flow(test_z)
    recovered_z, inverse_logdet = flow.inverse(test_x)
print("maximum inverse error:", float((test_z - recovered_z).abs().max().cpu()))
print("maximum log-det cancellation error:", float((forward_logdet + inverse_logdet).abs().max().cpu()))

# Record the actual output after every learned coupling layer, then apply the
# exact inverse layers in reverse order. This is not an interpolation: every
# frame is a real intermediate representation in the trained network.
with torch.no_grad():
    roundtrip = fixed_z[:500].clone()
    roundtrip_frames = [roundtrip.cpu().numpy()]
    roundtrip_labels = ["source z"]
    for layer_index, layer in enumerate(flow.layers, start=1):
        roundtrip, _ = layer(roundtrip)
        roundtrip_frames.append(roundtrip.cpu().numpy())
        roundtrip_labels.append(f"forward coupling {layer_index}")
    for reverse_index, layer in enumerate(reversed(flow.layers), start=1):
        roundtrip, _ = layer.inverse(roundtrip)
        roundtrip_frames.append(roundtrip.cpu().numpy())
        label = f"inverse coupling {reverse_index}"
        if reverse_index == len(flow.layers):
            label += " — recovered z"
        roundtrip_labels.append(label)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(target_np[:, 0], target_np[:, 1], s=5, alpha=0.10, c="black", label="target data")
roundtrip_marks = ax.scatter([], [], s=9, alpha=0.50, c="tab:blue", label="tracked samples")
ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2); ax.legend(loc="upper right")
def update_roundtrip(frame_index):
    roundtrip_marks.set_offsets(roundtrip_frames[frame_index])
    ax.set_title(roundtrip_labels[frame_index])
    return (roundtrip_marks,)
roundtrip_animation = FuncAnimation(fig, update_roundtrip, frames=len(roundtrip_frames), interval=900, blit=False)
roundtrip_path = ROOT / "fallbacks" / f"03_realnvp_roundtrip{artifact_suffix}.gif"
roundtrip_animation.save(roundtrip_path, writer=PillowWriter(fps=1))
plt.close(fig)
display(HTML(roundtrip_animation.to_jshtml()))
print("saved", roundtrip_path)

## 7. Why a flow does not compress dimensionality

Every flow layer is a bijection between spaces of equal dimension. Its Jacobian must be square and nonsingular. A VAE may map x ∈ R⁷⁸⁴ to z ∈ R² because its encoder is not required to be invertible. A normalising flow over x ∈ R⁷⁸⁴ needs z ∈ R⁷⁸⁴.

In [ ]:
flow_8d = RealNVP(dim=8, layers=8, hidden=96).to(device)
z_8d = torch.randn(32, 8, device=device)
x_8d, _ = flow_8d(z_8d)
z_recovered, _ = flow_8d.inverse(x_8d)
print("source shape:", tuple(z_8d.shape))
print("target shape:", tuple(x_8d.shape))
print("square Jacobian for one example: 8 × 8")
print("inverse error:", float((z_8d - z_recovered).abs().max().detach().cpu()))
print("A 2D bottleneck would destroy invertibility; use a VAE when compression is the goal.")

## Interpretation check and optional extension

Relate every visible expansion or contraction to the sign of the log-determinant. Explain why a small inverse error is necessary but does not prove that the learned density matches the data.

**Optional extension:** add coupling layers, then compare held-out negative log-likelihood, inverse error, and training time.

Record the changed parameter, the fixed controls, the observed evidence, and one limitation.